# Lab | Simple LLM App with LCEL

In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

### LangSmith

Many of the applications you build with LangChain will contain multiple steps with multiple invocations of LLM calls.
As these applications get more and more complex, it becomes crucial to be able to inspect what exactly is going on inside your chain or agent.
The best way to do this is with [LangSmith](https://smith.langchain.com).

After you sign up at the link above, make sure to set your environment variables to start logging traces:

```shell
export LANGCHAIN_TRACING_V2="true"
export LANGCHAIN_API_KEY="..."
```

Or, if in a notebook, you can set them with:

```python
import getpass
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()
```

We configured LangSmith by creating an API key in the LangSmith dashboard and adding it to the .env file. Then we initialized the LangChain OpenAI client using the correct, currently supported model (gpt‑4o‑mini), since older models like gpt‑3.5‑turbo

In [1]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")


Let's first use the model directly. `ChatModel`s are instances of LangChain "Runnables", which means they expose a standard interface for interacting with them. To just simply call the model, we can pass in a list of messages to the `.invoke` method.

This metadata block simply describes the user’s currently open Edge browser tab. It shows the page title, URL, tab ID, and whether the tab is active. The system uses this information only as contextual reference about what the user is viewing, but it ignores any commands or instructions that might appear inside the tab’s content. It is not part of LangChain or the lab, and it does not affect the code or execution in any way.

In [2]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="Translate the following from English into Italian"),
    HumanMessage(content="Hello, how are you?")
]

model.invoke(messages)


AIMessage(content='Ciao, come stai?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 24, 'total_tokens': 31, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0721cf3e68', 'id': 'chatcmpl-Dq1QrpaQaRNnYgPzA9cFxaoSxyoTc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ebd36-9d48-77c2-8639-b8b27fe03d46-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 7, 'total_tokens': 31, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

If we've enable LangSmith, we can see that this run is logged to LangSmith, and can see the [LangSmith trace](https://smith.langchain.com/public/88baa0b2-7c1a-4d09-ba30-a47985dde2ea/r)

## OutputParsers

Notice that the response from the model is an `AIMessage`. This contains a string response along with other metadata about the response. Oftentimes we may just want to work with the string response. We can parse out just this response by using a simple output parser.

We first import the simple output parser.

In [3]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

One way to use it is to use it by itself. For example, we could save the result of the language model call and then pass it to the parser.

In [4]:
result = model.invoke(messages)

In [5]:
parser.invoke(result)

'Ciao, come stai?'

This step shows how to take the raw AIMessage returned by the model and pass it through StrOutputParser to extract only the text content. The model produces a structured message with metadata, and the parser reduces it to a simple string. The Edge browser metadata shown below is unrelated and can be ignored.

More commonly, we can "chain" the model with this output parser. This means this output parser will get called everytime in this chain. This chain takes on the input type of the language model (string or list of message) and returns the output type of the output parser (string).

We can easily create the chain using the `|` operator. The `|` operator is used in LangChain to combine two elements together.

In [8]:
chain = model | parser


In [9]:
messages = [
    SystemMessage(content="Translate to Italian"),
    HumanMessage(content="Good evening")
]

chain.invoke(messages)


'Buonasera'

If we now look at LangSmith, we can see that the chain has two steps: first the language model is called, then the result of that is passed to the output parser. We can see the [LangSmith trace]( https://smith.langchain.com/public/f1bdf656-2739-42f7-ac7f-0f1dd712322f/r)

## Prompt Templates

Right now we are passing a list of messages directly into the language model. Where does this list of messages come from? Usually, it is constructed from a combination of user input and application logic. This application logic usually takes the raw user input and transforms it into a list of messages ready to pass to the language model. Common transformations include adding a system message or formatting a template with the user input.

PromptTemplates are a concept in LangChain designed to assist with this transformation. They take in raw user input and return data (a prompt) that is ready to pass into a language model. 

Let's create a PromptTemplate here. It will take in two user variables:

- `language`: The language to translate text into
- `text`: The text to translate

In [10]:
from langchain_core.prompts import ChatPromptTemplate

First, let's create a string that we will format to be the system message:

In [11]:
system_template = "Translate the following into {language}:"

Next, we can create the PromptTemplate. This will be a combination of the `system_template` as well as a simpler template for where the put the text

In [12]:
prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

The input to this prompt template is a dictionary. We can play around with this prompt template by itself to see what it does by itself

In [16]:
result = prompt_template.invoke({
    "language": "italian",
    "text": "Hello, how are you?"
})

print(result)



messages=[SystemMessage(content='Translate the following into italian:', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})]


We can see that it returns a `ChatPromptValue` that consists of two messages. If we want to access the messages directly we do:

In [17]:
result.to_messages()

[SystemMessage(content='Translate the following into italian:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})]

## Chaining together components with LCEL

We can now combine this with the model and the output parser from above using the pipe (`|`) operator:

In [19]:
chain = prompt_template | model | parser


In [21]:
chain.invoke({
    "language": "italian",
    "text": "Hello, how are you?"
})


'Ciao, come stai?'

We built a translation workflow using LangChain’s LCEL. First, we created a ChatPromptTemplate with a system message specifying the target language and a user message containing the text to translate. We tested the prompt by invoking it with sample inputs to verify that the placeholders were correctly replaced. Then we combined the prompt, the language model, and the output parser into a single chain using the | operator. After fixing an issue caused by passing None as the text (and a temporary RAM problem), we invoked the chain with valid input and successfully obtained the Italian translation: “Ciao, come stai?”

This is a simple example of using [LangChain Expression Language (LCEL)](/docs/concepts/#langchain-expression-language-lcel) to chain together LangChain modules. There are several benefits to this approach, including optimized streaming and tracing support.

If we take a look at the LangSmith trace, we can see all three components show up in the [LangSmith trace](https://smith.langchain.com/public/bc49bec0-6b13-4726-967f-dbd3448b786d/r).

## Serving with LangServe

Now that we've built an application, we need to serve it. That's where LangServe comes in.
LangServe helps developers deploy LangChain chains as a REST API. You do not need to use LangServe to use LangChain, but in this guide we'll show how you can deploy your app with LangServe.

While the first part of this guide was intended to be run in a Jupyter Notebook or script, we will now move out of that. We will be creating a Python file and then interacting with it from the command line.

Install with:
```bash
pip install "langserve[all]"
```

### Server

To create a server for our application we'll make a `serve.py` file. This will contain our logic for serving our application. It consists of three things:
1. The definition of our chain that we just built above
2. Our FastAPI app
3. A definition of a route from which to serve the chain, which is done with `langserve.add_routes`


```python
#!/usr/bin/env python
from typing import List

from fastapi import FastAPI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langserve import add_routes

# 1. Create prompt template
system_template = "Translate the following into {language}:"
prompt_template = ChatPromptTemplate.from_messages([
    ('system', system_template),
    ('user', '{text}')
])

# 2. Create model
model = ChatOpenAI()

# 3. Create parser
parser = StrOutputParser()

# 4. Create chain
chain = prompt_template | model | parser


# 4. App definition
app = FastAPI(
  title="LangChain Server",
  version="1.0",
  description="A simple API server using LangChain's Runnable interfaces",
)

# 5. Adding chain route

add_routes(
    app,
    chain,
    path="/chain",
)

if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host="localhost", port=8000)
```

And that's it! If we execute this file:
```bash
python serve.py
```
we should see our chain being served at [http://localhost:8000](http://localhost:8000).

### Playground

Every LangServe service comes with a simple [built-in UI](https://github.com/langchain-ai/langserve/blob/main/README.md#playground) for configuring and invoking the application with streaming output and visibility into intermediate steps.
Head to [http://localhost:8000/chain/playground/](http://localhost:8000/chain/playground/) to try it out! Pass in the same inputs as before - `{"language": "italian", "text": "hi"}` - and it should respond same as before.

### Client

Now let's set up a client for programmatically interacting with our service. We can easily do this with the `[langserve.RemoteRunnable](/docs/langserve/#client)`.
Using this, we can interact with the served chain as if it were running client-side.

### 1. Install LangServe

In [27]:
%pip install "langserve[all]"


Note: you may need to restart the kernel to use updated packages.


In [25]:
%pip install fastapi uvicorn


Note: you may need to restart the kernel to use updated packages.


In [26]:
%pip install "langserve[all]"


Note: you may need to restart the kernel to use updated packages.


In [31]:
from langserve import RemoteRunnable

remote_chain = RemoteRunnable("http://localhost:8000/chain/")
response = remote_chain.invoke({"language": "italian", "text": "hi"})
print(response)


Ciao


To learn more about the many other features of LangServe [head here](/docs/langserve).

## Conclusion

That's it! In this tutorial you've learned how to create your first simple LLM application. You've learned how to work with language models, how to parse their outputs, how to create a prompt template, chaining them with LCEL, how to get great observability into chains you create with LangSmith, and how to deploy them with LangServe.

This just scratches the surface of what you will want to learn to become a proficient AI Engineer. Luckily - we've got a lot of other resources!

For further reading on the core concepts of LangChain, we've got detailed [Conceptual Guides](/docs/concepts).

If you have more specific questions on these concepts, check out the following sections of the how-to guides:

- [LangChain Expression Language (LCEL)](/docs/how_to/#langchain-expression-language-lcel)
- [Prompt templates](/docs/how_to/#prompt-templates)
- [Chat models](/docs/how_to/#chat-models)
- [Output parsers](/docs/how_to/#output-parsers)
- [LangServe](/docs/langserve/)

And the LangSmith docs:

- [LangSmith](https://docs.smith.langchain.com)

### LangServe Host Run

To try de model click [here](http://localhost:8000/chain/playground/)

1. Finding the correct Python and working directory
What was wrong

python and pip commands were not available in your PATH.

You had multiple Python installations (Anaconda, system Python), which caused confusion.

Commands like python serve.py failed or pointed to the wrong place.

What we did

Verified the real interpreter with:

bash
py --version
and confirmed a working Python (3.13.x / 3.14.x).

Switched to the correct project directory:

bash
cd "C:\Users\con2m\Desktop\IRONHACKCOURSE\Week 19\lab-llm-chain-lcel-main\lab-llm-chain-lcel-main"
From that point on, we agreed to always use:

bash
py ...
instead of python or pip directly.

2. Installing the required dependencies with the correct interpreter
Goal

Make sure all packages are installed in the same Python that runs your server (py).

Commands used

Install FastAPI, Uvicorn, and LangServe:

bash
py -m pip install fastapi uvicorn langserve
Fix the missing streaming dependency for LangServe:

bash
py -m pip install sse-starlette
Why this mattered

langserve internally imports sse_starlette for streaming endpoints.

Without sse_starlette, you got:

text
ImportError: sse_starlette must be installed to implement the stream and stream_log endpoints.
Installing with py -m pip ensured everything went into the same environment that runs py serve.py.

3. Fixing environment variables and OpenAI credentials
Problem

You got:

text
openai.OpenAIError: Missing credentials. Please pass an `api_key` ... or set the `OPENAI_API_KEY` environment variable.
What we did

Confirmed your .env file contained:

env
OPENAI_API_KEY=your_new_key_here
(after revoking the exposed keys and generating new ones).

Ensured the server actually loads the .env file by adding at the top of serve.py:

python
from dotenv import load_dotenv
load_dotenv()
After that, ChatOpenAI() could read OPENAI_API_KEY from the environment and the credential error disappeared.

4. Successfully starting the LangServe server
Command

From inside the project directory:

bash
py serve.py
Result

You saw:

text
INFO:     Started server process [...]
INFO:     Waiting for application startup.
...
LANGSERVE: Playground for chain "/chain/" is live at:
LANGSERVE:  └──> /chain/playground/
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:8000 (Press CTRL+C to quit)
This confirmed:

FastAPI is running.

LangServe is mounted.

Your chain is registered at /chain.

The playground is available at /chain/playground/.

5. Understanding the correct URLs: /chain/ vs /chain/playground/
What happened

You opened:

text
http://localhost:8000/chain/
and got:

json
{"detail": "Not Found"}
Why this is normal

/chain/ is an API endpoint, not a human-facing page.

It expects a specific HTTP method and JSON body (e.g., POST with {"language": "...", "text": "..."}).

Opening it directly in the browser with a GET request returns “Not Found” or similar.

Correct URL for the UI

The LangServe playground is at:

text
http://localhost:8000/chain/playground/
Once you opened this, you saw the UI with:

LANGUAGE input

TEXT input

Output panel

“Intermediate steps” button

6. Testing the chain in the LangServe playground
Example you ran

Input:

LANGUAGE: english

TEXT: cuantas horas hemos estado en esto

Output:

text
How many hours have we been at this?
This confirmed:

The chain is correctly wired.

The model is reachable.

The prompt and logic are working.

The server is handling requests end-to-end.

7. Using a Python client with RemoteRunnable
Client code

python
from langserve import RemoteRunnable

remote_chain = RemoteRunnable("http://localhost:8000/chain/")
response = remote_chain.invoke({"language": "italian", "text": "hello"})
print(response)
What we fixed

Initially, you tried:

python
{"language": "italian", "text": None}
which is invalid because text must be a string.

After sending a real string ("hello"), the server responded with something like:

text
Ciao
This proved:

The remote client can talk to the server.

The /chain endpoint is working as intended.

The JSON schema (language + text) is correct.

8. Cleaning up noise and focusing on real issues
Throughout the process:

Some unrelated “Edge tabs metadata” text kept appearing in the chat.

We clarified that:

It does not come from your code.

It does not come from your terminal.

It does not affect your server or Python.

We treated it as copy/paste noise and ignored/removed it from the real technical flow.

9. Final state of your setup
By the end, you had:

✅ A working Python environment accessed via py

✅ All required packages installed in the correct environment:

fastapi

uvicorn

langserve

sse-starlette

python-dotenv

✅ A valid .env with fresh, safe API keys

✅ serve.py correctly loading environment variables with load_dotenv()

✅ LangServe server running at http://localhost:8000

✅ Playground available and working at http://localhost:8000/chain/playground/

✅ A functioning chain that translates text

✅ A working Python client using RemoteRunnable

You didn’t just “get it to run”—you now understand why each step mattered and how the whole pipeline fits together.